# Verificación de gen_1 sobre la poda ganadora (asesinos_tuneado_2sd, 358 vars)

`gen_1 = ctrx_quarter + cpayroll_trx_min3` -- el único sobreviviente del algoritmo genético overnight -- ya
demostró sumar +4.29M sobre la poda v1 vieja (110 vars, 439.2M -> 443.52M). Ahora que tenemos una poda mucho
mejor (`asesinos_tuneado_2sd`, 358 vars, 462.44M confirmado con 10 semillas), chequeamos si sigue sumando ahí,
antes de decidir si vale la pena correr un algoritmo genético nuevo sobre este pool más grande.

### Setup y carga de datos

In [1]:
rm(list = ls())
gc()

require("data.table")
require("lightgbm")

PARAM_carpeta_datasets <- "C:/Users/Laura/Documents/DMEyF/datasets/"
PARAM_carpeta_exp <- "C:/Users/Laura/Documents/DMEyF/exp/"

t0 <- Sys.time()
dataset <- fread(paste0(PARAM_carpeta_datasets, "competencia_01_con_lags_fe_historico_rankeado.csv.gz"))
cat("Dataset cargado:", nrow(dataset), "filas x", ncol(dataset), "columnas en",
    round(as.numeric(Sys.time()-t0, units="secs"),1), "seg\n")

dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,653147,34.9,1447520,77.4,1091942,58.4
Vcells,1199194,9.2,8388608,64.0,1928501,14.8


Caricamento del pacchetto richiesto: data.table

Caricamento del pacchetto richiesto: lightgbm



Dataset cargado: 983061 filas x 1622 columnas en 200.4 seg


### Reconstruyo las compuestas (por si alguna sobrevivió en la lista ganadora) y gen_1

In [2]:
agregar_si_existen <- function(dt, nombre_nueva, columnas, fun_combinacion) {
  presentes <- columnas[columnas %in% colnames(dt)]
  if (length(presentes) < 2) return(invisible(NULL))
  dt[, (nombre_nueva) := fun_combinacion(.SD), .SDcols = presentes]
}
suma_filas <- function(sd) Reduce(`+`, sd)

agregar_si_existen(dataset, "consumo_tarjetas_total", c("mtarjeta_visa_consumo", "mtarjeta_master_consumo"), suma_filas)
agregar_si_existen(dataset, "consumototal_tarjetas", c("Visa_mconsumototal", "Master_mconsumototal"), suma_filas)
agregar_si_existen(dataset, "saldo_tarjetas_total", c("Visa_msaldototal", "Master_msaldototal"), suma_filas)
agregar_si_existen(dataset, "limite_tarjetas_total", c("Visa_mlimitecompra", "Master_mlimitecompra"), suma_filas)
agregar_si_existen(dataset, "pagado_tarjetas_total", c("Visa_mpagado", "Master_mpagado"), suma_filas)
agregar_si_existen(dataset, "pagominimo_tarjetas_total", c("Visa_mpagominimo", "Master_mpagominimo"), suma_filas)
agregar_si_existen(dataset, "descuentos_tarjetas_total",
                    c("mcajeros_propios_descuentos", "mtarjeta_visa_descuentos", "mtarjeta_master_descuentos"), suma_filas)

if (all(c("consumototal_tarjetas", "limite_tarjetas_total") %in% colnames(dataset))) {
  dataset[, utilizacion_tarjetas := ifelse(limite_tarjetas_total == 0, 0,
                                            consumototal_tarjetas / limite_tarjetas_total)]
}
if (all(c("Visa_status", "Master_status") %in% colnames(dataset))) {
  dataset[, tarjetas_activas_count := as.integer(Visa_status == 0) + as.integer(Master_status == 0)]
}

agregar_si_existen(dataset, "ingresos_payroll_total", c("mpayroll", "mpayroll2"), suma_filas)
agregar_si_existen(dataset, "prestamos_sin_personales_total",
                    c("mprestamos_prendarios", "mprestamos_hipotecarios"), suma_filas)
agregar_si_existen(dataset, "inversiones_totales",
                    c("minversion1_pesos", "minversion1_dolares", "minversion2"), suma_filas)
agregar_si_existen(dataset, "plazo_fijo_total", c("mplazo_fijo_dolares", "mplazo_fijo_pesos"), suma_filas)

if (all(c("mtransferencias_recibidas", "mtransferencias_emitidas") %in% colnames(dataset))) {
  dataset[, transferencias_netas := mtransferencias_recibidas - mtransferencias_emitidas]
}
agregar_si_existen(dataset, "comisiones_totales_manual", c("mcomisiones_mantenimiento", "mcomisiones_otras"), suma_filas)

# la variable descubierta por el genetico
dataset[, gen_1 := ctrx_quarter + cpayroll_trx_min3]

cat("Compuestas y gen_1 reconstruidas\n")

Compuestas y gen_1 reconstruidas


### Cargo la lista ganadora (asesinos_tuneado_2sd, 358 vars) y armo las dos variantes a comparar

In [3]:
campos_ganadora <- fread(paste0(PARAM_carpeta_exp, "campos_canaritos_asesinos_2sd.txt"))$variable
campos_ganadora <- campos_ganadora[campos_ganadora %in% colnames(dataset)]

campos_sin_genetico <- campos_ganadora
campos_con_genetico <- c(campos_ganadora, "gen_1")

cat("Sin genetico:", length(campos_sin_genetico), "vars\n")
cat("Con genetico:", length(campos_con_genetico), "vars\n")

Sin genetico: 358 vars
Con genetico: 359 vars


### Split train / future

In [4]:
dataset_train <- dataset[foto_mes %in% c(202103, 202104)]
dfuture <- dataset[foto_mes == 202106]
clase_ternaria_future <- dfuture[, clase_ternaria]

cat("train:", nrow(dataset_train), "filas | future:", nrow(dfuture), "filas\n")

train: 326184 filas | future: 164114 filas


### Hiperparámetros manuales (min_data_in_leaf = 4000, mismo track de siempre)

In [5]:
param_manual <- list(
  boosting = "gbdt",
  objective = "binary",
  metric = "auc",
  first_metric_only = TRUE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  verbosity = -100,

  learning_rate = 0.02,
  feature_fraction = 0.1541961,
  feature_fraction_bynode = 1,
  min_data_in_leaf = 4000,
  num_leaves = 1684,
  max_depth = -1,
  lambda_l1 = 1.520327,
  lambda_l2 = 7.548036,
  min_gain_to_split = 0.2158453,
  max_delta_step = 0,
  bagging_fraction = 0.8425105,
  bagging_freq = 6,
  min_sum_hessian_in_leaf = 0.001,
  pos_bagging_fraction = 1,
  neg_bagging_fraction = 1,
  extra_trees = FALSE,
  path_smooth = 0,
  cat_l2 = 10,
  cat_smooth = 10,
  max_cat_to_onehot = 4,
  top_rate = 0.2,
  other_rate = 0.1,
  use_missing = TRUE,
  is_unbalance = FALSE,
  num_iterations = 600,
  early_stopping_rounds = 0,
  max_bin = 31,
  scale_pos_weight = 0.5,
  force_row_wise = TRUE
)

SEMILLAS_SCREENING <- c(585581, 249449, 655649)
SEMILLAS_10 <- c(585581, 249449, 655649, 503039, 798191, 112909, 334667, 421303, 667021, 887923)
CORTE_FIJO <- 12000

### Función de evaluación

In [6]:
evaluar <- function(campos, semillas) {
  matriz_train <- data.matrix(dataset_train[, campos, with = FALSE])
  matriz_future <- data.matrix(dfuture[, campos, with = FALSE])

  ganancias <- c()
  for (semilla in semillas) {
    param_semilla <- param_manual
    param_semilla$seed <- semilla

    dtrain <- lgb.Dataset(data = matriz_train, label = dataset_train$clase01, free_raw_data = FALSE)
    modelo <- lgb.train(data = dtrain, param = param_semilla, verbose = -1)
    prediccion <- predict(modelo, matriz_future)

    tbl <- data.table(prob = prediccion, clase_ternaria = clase_ternaria_future)
    setorder(tbl, -prob)
    tbl[, gan_individual := ifelse(clase_ternaria == "BAJA+2", 1072500, -27500)]
    ganancia <- sum(tbl[1:CORTE_FIJO, gan_individual])

    ganancias <- c(ganancias, ganancia)
    cat("    semilla", semilla, "-> ganancia", ganancia, "\n")

    rm(dtrain, modelo)
    gc(verbose = FALSE)
  }
  ganancias
}

### Paso 1: screening con 3 semillas

In [7]:
cat("=== Sin genetico (358 vars) -- screening ===\n")
gan_sin <- evaluar(campos_sin_genetico, SEMILLAS_SCREENING)
cat("media sin genetico:", mean(gan_sin), "\n\n")

cat("=== Con genetico (359 vars, + gen_1) -- screening ===\n")
gan_con <- evaluar(campos_con_genetico, SEMILLAS_SCREENING)
cat("media con genetico:", mean(gan_con), "\n")

cat("\nDiferencia (screening):", mean(gan_con) - mean(gan_sin), "\n")
cat("Referencia: sobre la poda v1 vieja (110 vars) gen_1 sumaba +4.29M (10 semillas)\n")

=== Sin genetico (358 vars) -- screening ===
    semilla 585581 -> ganancia 460900000 
    semilla 249449 -> ganancia 463100000 
    semilla 655649 -> ganancia 468600000 
media sin genetico: 464200000 

=== Con genetico (359 vars, + gen_1) -- screening ===
    semilla 585581 -> ganancia 463100000 
    semilla 249449 -> ganancia 465300000 
    semilla 655649 -> ganancia 456500000 
media con genetico: 461633333 

Diferencia (screening): -2566667 
Referencia: sobre la poda v1 vieja (110 vars) gen_1 sumaba +4.29M (10 semillas)


### Paso 2: si se ve bien, confirmar con las 10 semillas completas

In [8]:
cat("=== Sin genetico (358 vars) -- 10 semillas ===\n")
gan_sin_10 <- evaluar(campos_sin_genetico, SEMILLAS_10)
cat("media sin genetico (10 semillas):", mean(gan_sin_10), "desvio:", sd(gan_sin_10), "\n\n")

cat("=== Con genetico (359 vars) -- 10 semillas ===\n")
gan_con_10 <- evaluar(campos_con_genetico, SEMILLAS_10)
cat("media con genetico (10 semillas):", mean(gan_con_10), "desvio:", sd(gan_con_10), "\n")

cat("\nDiferencia (10 semillas):", mean(gan_con_10) - mean(gan_sin_10), "\n")

=== Sin genetico (358 vars) -- 10 semillas ===
    semilla 585581 -> ganancia 460900000 
    semilla 249449 -> ganancia 463100000 
    semilla 655649 -> ganancia 468600000 
    semilla 503039 -> ganancia 463100000 
    semilla 798191 -> ganancia 465300000 
    semilla 112909 -> ganancia 466400000 
    semilla 334667 -> ganancia 459800000 
    semilla 421303 -> ganancia 457600000 
    semilla 667021 -> ganancia 464200000 
    semilla 887923 -> ganancia 455400000 
media sin genetico (10 semillas): 462440000 desvio: 4056600 

=== Con genetico (359 vars) -- 10 semillas ===
    semilla 585581 -> ganancia 463100000 
    semilla 249449 -> ganancia 465300000 
    semilla 655649 -> ganancia 456500000 
    semilla 503039 -> ganancia 474100000 
    semilla 798191 -> ganancia 465300000 
    semilla 112909 -> ganancia 456500000 
    semilla 334667 -> ganancia 467500000 
    semilla 421303 -> ganancia 464200000 
    semilla 667021 -> ganancia 465300000 
    semilla 887923 -> ganancia 4.62e+08 
media